In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Explode
data = [(1, 'syed sadiq',['reading', 'working', 'seeing movies', 'coding']), (2, 'syed saleema sulthana', ['hosehold works', 'job', 'teaching']), (3, 'syed afzal miya', ['movies', 'house hold works'])]

schema = StructType([
            StructField('id', IntegerType()), 
            StructField('name', StringType()),
            StructField('hobbies', ArrayType(StringType()))
            ])

df = spark.createDataFrame(data, schema)
df = df.withColumn('hobby', explode('hobbies'))
display(df)

id,name,hobbies,hobby
1,syed sadiq,"List(reading, working, seeing movies, coding)",reading
1,syed sadiq,"List(reading, working, seeing movies, coding)",working
1,syed sadiq,"List(reading, working, seeing movies, coding)",seeing movies
1,syed sadiq,"List(reading, working, seeing movies, coding)",coding
2,syed saleema sulthana,"List(hosehold works, job, teaching)",hosehold works
2,syed saleema sulthana,"List(hosehold works, job, teaching)",job
2,syed saleema sulthana,"List(hosehold works, job, teaching)",teaching
3,syed afzal miya,"List(movies, house hold works)",movies
3,syed afzal miya,"List(movies, house hold works)",house hold works


In [0]:

## Split
data1 = [
            (1, 'syed sadiq','reading, working, seeing movies, coding'), 
            (2, 'syed saleema sulthana', 'hosehold works, job, teaching'), 
            (3, 'syed afzal miya', 'movies, house hold works')
        ]

schema = StructType([
            StructField('id', IntegerType()), 
            StructField('name', StringType()),
            StructField('hobbies', StringType())
            ])

df = spark.createDataFrame(data1, schema)
df = df.withColumn('hobby', explode(split('hobbies', ',')))
display(df)

id,name,hobbies,hobby
1,syed sadiq,"reading, working, seeing movies, coding",reading
1,syed sadiq,"reading, working, seeing movies, coding",working
1,syed sadiq,"reading, working, seeing movies, coding",seeing movies
1,syed sadiq,"reading, working, seeing movies, coding",coding
2,syed saleema sulthana,"hosehold works, job, teaching",hosehold works
2,syed saleema sulthana,"hosehold works, job, teaching",job
2,syed saleema sulthana,"hosehold works, job, teaching",teaching
3,syed afzal miya,"movies, house hold works",movies
3,syed afzal miya,"movies, house hold works",house hold works


In [0]:

## Split
data1 = [
            (1, 'syed sadiq','reading, working, seeing movies, coding'), 
            (2, 'syed saleema sulthana', 'hosehold works, job, teaching'), 
            (3, 'syed afzal miya', 'movies, house hold works')
        ]

schema = StructType([
            StructField('id', IntegerType()), 
            StructField('name', StringType()),
            StructField('hobbies', StringType())
            ])

df = spark.createDataFrame(data1, schema)
df = df.withColumn('hobby', explode(split(col('hobbies'), ',')))
df = df.withColumn('hobby_array', array(col('hobby'), col('hobbies')))
df = df.withColumn('final', explode(col('hobby_array')))
df = df.withColumn('final', explode(split(col('final'), ',')))
df = df.withColumn('Is_Working', when(col('final').isin('hosehold works', 'teaching', ' job'), lit('true')).
                                 when(col('final') =='coding', lit('Null')).otherwise(lit('false')))
display(df.distinct())

id,name,hobbies,hobby,hobby_array,final,Is_Working
1,syed sadiq,"reading, working, seeing movies, coding",working,"List( working, reading, working, seeing movies, coding)",reading,null
1,syed sadiq,"reading, working, seeing movies, coding",working,"List( working, reading, working, seeing movies, coding)",coding,null
1,syed sadiq,"reading, working, seeing movies, coding",coding,"List( coding, reading, working, seeing movies, coding)",working,null
1,syed sadiq,"reading, working, seeing movies, coding",coding,"List( coding, reading, working, seeing movies, coding)",coding,null
1,syed sadiq,"reading, working, seeing movies, coding",reading,"List(reading, reading, working, seeing movies, coding)",working,null
1,syed sadiq,"reading, working, seeing movies, coding",reading,"List(reading, reading, working, seeing movies, coding)",seeing movies,null
1,syed sadiq,"reading, working, seeing movies, coding",working,"List( working, reading, working, seeing movies, coding)",working,null
1,syed sadiq,"reading, working, seeing movies, coding",seeing movies,"List( seeing movies, reading, working, seeing movies, coding)",seeing movies,null
1,syed sadiq,"reading, working, seeing movies, coding",reading,"List(reading, reading, working, seeing movies, coding)",reading,null
1,syed sadiq,"reading, working, seeing movies, coding",working,"List( working, reading, working, seeing movies, coding)",seeing movies,null


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

dataa2 = [
        (1, 'syed sadiq', {'hair' : 'black', 'skintone' : 'fair', 'eyes' : 'brown'}),
        (2, 'syed saleema sulthana', {'hair' : 'brown', 'skintone' : 'fair', 'eyes' : 'black'}),
        (3, 'syed afzal miya', {'hair' : 'white and black', 'skintone' : 'fair', 'remarks' : 'king of my home my real king i love my father he is strong than me'})
]

schema = StructType([
            StructField('id', IntegerType()), 
            StructField('name', StringType()),
            StructField('charectristics', MapType(StringType(), StringType()))
            ])


# df = spark.createDataFrame(dataa2, schema)
# df = (df.withColumn('charectristics_eyes', df.charectristics.getItem('eyes'))
#         .withColumn('charectristics_hair', df.charectristics.getItem('hair'))
#         .withColumn('charectristics_skintone', df.charectristics.getItem('skintone')).
#         withColumn('charectristics_remarks', df.charectristics.getItem('remarks')))
# df.display(truncae= True)


df = spark.createDataFrame(dataa2, schema)
#df = df.select('id', 'name', explode(df.charectristics))
df = df.withColumn('KeyMapping', map_keys(df.charectristics))
df = df.withColumn('ValueMapping', map_values(df.charectristics))
df.printSchema()
df.display()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- charectristics: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- KeyMapping: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ValueMapping: array (nullable = true)
 |    |-- element: string (containsNull = true)



id,name,charectristics,KeyMapping,ValueMapping
1,syed sadiq,"Map(hair -> black, skintone -> fair, eyes -> brown)","List(hair, skintone, eyes)","List(black, fair, brown)"
2,syed saleema sulthana,"Map(hair -> brown, skintone -> fair, eyes -> black)","List(hair, skintone, eyes)","List(brown, fair, black)"
3,syed afzal miya,"Map(hair -> white and black, skintone -> fair, remarks -> king of my home my real king i love my father he is strong than me)","List(hair, skintone, remarks)","List(white and black, fair, king of my home my real king i love my father he is strong than me)"


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

data3 = [
    (1, 'A', 20, 'TEACHING'),
    (2, 'B', 25, 'ACCOUNTIG'),
    (3, 'C', 26, 'NON TEACHING')
]

schema3 = StructType([
            StructField('EMP_ID', IntegerType()), 
            StructField('EMP_NAME', StringType()),
            StructField('EMP_AGE', IntegerType()),
            StructField('EMP_DEPT', StringType())
            ])


data4 = [
    (1, 2000),
    (2, 3000),
]

schema4 = StructType([
            StructField('EMP_ID', IntegerType()), 
            StructField('EMP_SAL', IntegerType())
            ])

    
EMP_DF = spark.createDataFrame(data3, schema3)
display(EMP_DF)

EMP_SAL = spark.createDataFrame(data4, schema4)
display(EMP_SAL)

INNER_df = EMP_DF.join(
                EMP_SAL, 
                on=EMP_DF.EMP_ID == EMP_SAL.EMP_ID,
                how='inner').withColumn('TypeofJoin', lit('INNER'))

display(INNER_df)


LEFT_df = EMP_DF.join(
                EMP_SAL, 
                on=EMP_DF.EMP_ID == EMP_SAL.EMP_ID,
                how= 'left').withColumn('TypeofJoin', lit('LEFT'))

display(LEFT_df)

RIGHT_DF = EMP_DF.join(
                EMP_SAL, 
                on=EMP_DF.EMP_ID == EMP_SAL.EMP_ID,
                how= 'right').withColumn('TypeofJoin', lit('RIGHT'))

display(RIGHT_DF)


FULL_OUTER_JOIN = EMP_DF.join(
                EMP_SAL, 
                on=EMP_DF.EMP_ID == EMP_SAL.EMP_ID,
                how= 'full_outer').withColumn('TypeofJoin', lit('FULLOUTER'))

display(FULL_OUTER_JOIN)

INNER_df.unionAll(LEFT_df).unionAll(RIGHT_DF).unionAll(FULL_OUTER_JOIN).display()


display(help(FULL_OUTER_JOIN))

EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT
1,A,20,TEACHING
2,B,25,ACCOUNTIG
3,C,26,NON TEACHING


EMP_ID,EMP_SAL
1,2000
2,3000


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,EMP_ID,EMP_SAL,TypeofJoin
1,A,20,TEACHING,1,2000,INNER
2,B,25,ACCOUNTIG,2,3000,INNER


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,EMP_ID,EMP_SAL,TypeofJoin
1,A,20,TEACHING,1,2000,LEFT
2,B,25,ACCOUNTIG,2,3000,LEFT
3,C,26,NON TEACHING,null,null,LEFT


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,EMP_ID,EMP_SAL,TypeofJoin
1,A,20,TEACHING,1,2000,RIGHT
2,B,25,ACCOUNTIG,2,3000,RIGHT


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,EMP_ID,EMP_SAL,TypeofJoin
1,A,20,TEACHING,1,2000,FULLOUTER
2,B,25,ACCOUNTIG,2,3000,FULLOUTER
3,C,26,NON TEACHING,null,null,FULLOUTER


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,EMP_ID,EMP_SAL,TypeofJoin
1,A,20,TEACHING,1,2000,INNER
2,B,25,ACCOUNTIG,2,3000,INNER
1,A,20,TEACHING,1,2000,LEFT
2,B,25,ACCOUNTIG,2,3000,LEFT
3,C,26,NON TEACHING,null,null,LEFT
1,A,20,TEACHING,1,2000,RIGHT
2,B,25,ACCOUNTIG,2,3000,RIGHT
1,A,20,TEACHING,1,2000,FULLOUTER
2,B,25,ACCOUNTIG,2,3000,FULLOUTER
3,C,26,NON TEACHING,null,null,FULLOUTER


Help on DataFrame in module pyspark.sql.connect.dataframe object:

class DataFrame(pyspark.sql.dataframe.DataFrame)
 |  DataFrame(plan: pyspark.sql.connect.plan.LogicalPlan, session: 'SparkSession') -> 'DataFrame'
 |
 |  Method resolution order:
 |      DataFrame
 |      pyspark.sql.dataframe.DataFrame
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __dir__(self) -> List[str]
 |      Examples
 |      --------
 |      >>> from pyspark.sql.functions import lit
 |
 |      Create a dataframe with a column named 'id'.
 |
 |      >>> df = spark.range(3)
 |      >>> [attr for attr in dir(df) if attr[0] == 'i'][:7] # Includes column id
 |      ['id', 'inputFiles', 'intersect', 'intersectAll', 'isEmpty', 'isLocal', 'isStreaming']
 |
 |      Add a column named 'i_like_pancakes'.
 |
 |      >>> df = df.withColumn('i_like_pancakes', lit(1))
 |      >>> [attr for attr in dir(df) if attr[0] == 'i'][:7] # Includes columns i_like_pancakes, id
 |      ['i_like_pancakes', 'id', 'inputFiles',

In [0]:
from pyspark.sql.functions import *
from  pyspark.sql.types import *
from pyspark.sql.window import *

data = [
    (1, 'John Doe', 30, 'Engineering'),
    (4, 'Remo', 25, 'Engineering'),
    (5, 'RAVI', 25, 'Engineering'),
    (7, 'DESH', 24, 'Engineering'),
    (6, 'KUMAR', 26, 'Engineering'),
    (2, 'Jane Smith', 28, 'Marketing'),
    (3, 'Alice Johnson', 35, 'Finance')
]

schema = StructType([
    StructField('EMP_ID', IntegerType()),
    StructField('EMP_NAME', StringType()),
    StructField('EMP_AGE', IntegerType()),
    StructField('EMP_DEPT', StringType())
])

rows = []
for emp in data:
    rows.append(emp)

df = spark.createDataFrame(rows, schema)
df = df.withColumn('row_num', row_number().over(Window.partitionBy('EMP_DEPT').orderBy(col('EMP_AGE').desc()))).\
        withColumn('WindowType', lit('ROW_NUMBER'))
display(df)
df = df.withColumn('row_num', rank().over(Window.partitionBy('EMP_DEPT').orderBy(col('EMP_AGE').desc()))).\
        withColumn('WindowType', lit('RANK'))
display(df)
df = df.withColumn('row_num', dense_rank().over(Window.partitionBy('EMP_DEPT').orderBy(col('EMP_AGE').desc()))).\
        withColumn('WindowType', lit('dense_RANK'))
display(df)
df = df.withColumn('row_num', lead('EMP_AGE', 1, 0).over(Window.partitionBy('EMP_DEPT').orderBy(col('EMP_AGE').desc()))).\
        withColumn('WindowType', lit('LEAD'))
display(df)
df = df.withColumn('row_num', lag('EMP_AGE', 1, 0).over(Window.partitionBy('EMP_DEPT').orderBy(col('EMP_AGE').desc()))).\
        withColumn('WindowType', lit('lag'))
display(df)

EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,row_num,WindowType
1,John Doe,30,Engineering,1,ROW_NUMBER
6,KUMAR,26,Engineering,2,ROW_NUMBER
4,Remo,25,Engineering,3,ROW_NUMBER
5,RAVI,25,Engineering,4,ROW_NUMBER
7,DESH,24,Engineering,5,ROW_NUMBER
3,Alice Johnson,35,Finance,1,ROW_NUMBER
2,Jane Smith,28,Marketing,1,ROW_NUMBER


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,row_num,WindowType
1,John Doe,30,Engineering,1,RANK
6,KUMAR,26,Engineering,2,RANK
4,Remo,25,Engineering,3,RANK
5,RAVI,25,Engineering,3,RANK
7,DESH,24,Engineering,5,RANK
3,Alice Johnson,35,Finance,1,RANK
2,Jane Smith,28,Marketing,1,RANK


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,row_num,WindowType
1,John Doe,30,Engineering,1,dense_RANK
6,KUMAR,26,Engineering,2,dense_RANK
4,Remo,25,Engineering,3,dense_RANK
5,RAVI,25,Engineering,3,dense_RANK
7,DESH,24,Engineering,4,dense_RANK
3,Alice Johnson,35,Finance,1,dense_RANK
2,Jane Smith,28,Marketing,1,dense_RANK


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,row_num,WindowType
1,John Doe,30,Engineering,26,LEAD
6,KUMAR,26,Engineering,25,LEAD
4,Remo,25,Engineering,25,LEAD
5,RAVI,25,Engineering,24,LEAD
7,DESH,24,Engineering,0,LEAD
3,Alice Johnson,35,Finance,0,LEAD
2,Jane Smith,28,Marketing,0,LEAD


EMP_ID,EMP_NAME,EMP_AGE,EMP_DEPT,row_num,WindowType
1,John Doe,30,Engineering,0,lag
6,KUMAR,26,Engineering,30,lag
4,Remo,25,Engineering,26,lag
5,RAVI,25,Engineering,25,lag
7,DESH,24,Engineering,25,lag
3,Alice Johnson,35,Finance,0,lag
2,Jane Smith,28,Marketing,0,lag
